In [1]:
import torch
import numpy as np 
import os 
from PIL import Image
import pandas as pd 
import json
import ast
import cv2
import h5py


import torch 
from torch.utils.data import Dataset
from torchvision import transforms
import collections


In [2]:

def resize_keypoints(keypoints, original_size, new_size):
    """
    Resize keypoints to match the new image size.

    Args:
        keypoints (list of tuple): List of keypoint coordinates [(x, y), ...]. for One person
        original_size (tuple): Original image size (H, W).
        new_size (tuple): Target image size (H, W).

    Returns:     
        list of tuple: Resized keypoint coordinates [(x', y'), ...]. for one person
    """
    orig_h, orig_w = original_size
    new_h, new_w = new_size
    scale_h, scale_w = new_h / orig_h, new_w / orig_w

    resized_keypoints = [(x * scale_w, y * scale_h) for x, y in keypoints]
    return resized_keypoints


limbs_pairs_person  = [
    (0, 5),   # Nose → Left Shoulder
    (0, 6),   # Nose → Right Shoulder
    (0, 1),   # Nose → Left Eye
    (0, 2),   # Nose → Right Eye
    (1, 3),   # Left Eye → Left Ear
    (2, 4),   # Right Eye → Right Ear
    (5, 6),   # Left Shoulder → Right Shoulder
    (5, 7),   # Left Shoulder → Left Elbow
    (7, 9),   # Left Elbow → Left Wrist
    (6, 8),   # Right Shoulder → Right Elbow
    (8, 10),  # Right Elbow → Right Wrist
    (5, 11),  # Left Shoulder → Left Hip
    (6, 12),  # Right Shoulder → Right Hip
    (11, 12), # Left Hip → Right Hip
    (11, 13), # Left Hip → Left Knee
    (13, 15), # Left Knee → Left Ankle
    (12, 14), # Right Hip → Right Knee
    (14, 16)  # Right Knee → Right Ankle
    ]
    
import numpy as np

def build_limb_vector_for_skeleton(gt_keypoints, bone_mapping):
    """
    Given a list of keypoints and a bone mapping, returns:
      1) limb_tensor:  A single tensor of shape (4 * L), 
         where L is the number of limbs. Each limb contributes [x1, y1, x2, y2].
      2) adjacency:    A (N x N) tensor indicating which joints are connected. 

    Args:
      gt_keypoints (list of tuples): [(x0, y0), (x1, y1), ..., (xN-1, yN-1)]
      bone_mapping (list of tuples): [(j1, j2), (j3, j4), ...]
        Each (j1, j2) indicates a limb between joint j1 and j2.

    Returns:
      dict with:
        - "limb_tensor": torch.Tensor of shape (4*L,)
        - "adjacency":   torch.Tensor of shape (N, N)
    """

    # 1) Concatenate limb endpoints into a single list
    #    [x1, y1, x2, y2, x3, y3, x4, y4, ...]
    limb_coords = []
    for (j1, j2) in bone_mapping:
        x1, y1 = gt_keypoints[j1]
        x2, y2 = gt_keypoints[j2]
        limb_coords.extend([x1, y1, x2, y2])

    # Convert to a PyTorch tensor (shape: [4*L])
    limb_tensor = torch.tensor(limb_coords, dtype=torch.float32)

    # 2) Build an NxN adjacency matrix as a tensor
    N = len(bone_mapping)
    adjacency = torch.zeros((N, N), dtype=torch.float32)
    for (j1, j2) in bone_mapping:
        adjacency[j1, j2] = 1.0
        adjacency[j2, j1] = 1.0  # if undirected

    return {
        "limb_tensor": limb_tensor,  # shape (4*L,)
        "adjacency":   adjacency     # shape (N, N)
    }




In [ ]:
    
    # VGAF transforms
general_transforms = {
        'train': transforms.Compose([
            #transforms.RandomResizedCrop(size=(224, 224)),
           # transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]),
        'val': transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]),
    }

In [ ]:

def SaveSkeletons(data_folder_src,csv_file_src, data_folder_dst, csv_file_dst, score=0.1):
    # create dir save data
    os.makedirs(data_folder_dst, exist_ok=True)
    
    with open(os.path.join(csv_file_src), 'r') as file:
        annotations = json.load(file)

    keep_files_names =[]

    for index in range(len(annotations)):
        annotation =   annotations[index]
        label_emotion = annotation['label_emotion']
        file_name = annotation['file_name']
        #label_emotion = annotation['emotion_label']
        persons = annotation["persons"]#[0]
        faces= annotation.get("faces", [])
         # Load and process the image
        image_context = Image.open(os.path.join( data_folder_src, annotation['file_name']))
        if image_context.mode != 'RGB':
            image_context = image_context.convert('RGB')
        original_size = image_context.size
        new_size = (224, 224)
        image_context = image_context.resize(new_size, Image.LANCZOS)
        
        if not persons['pose']:
            keypoints_list = [] # format (y,  x)
        else:
            keypoints_list = [[ kp[:2] if kp[-1] >= score else (-1, -1) for kp in keypoints ]  for keypoints in persons['pose'][0] ]
        
        if len(keypoints_list)!=0:
            # Iterate through each person's keypoints
            keypoints_list_resized = [resize_keypoints(person_kp, original_size, new_size) for person_kp in keypoints_list]

        #gt kpts 
        sekeleton_person_gt = []
        adjacency_person_gt = []
        for pers_id in range(len(keypoints_list_resized)):
            skeleton_data = build_limb_vector_for_skeleton(keypoints_list_resized[pers_id], limbs_pairs_person)
            sekeleton_person_gt.append(skeleton_data["limb_tensor"])
            adjacency_person_gt.append(skeleton_data["adjacency"])
       
        output_person_kp = data_folder_dst+'/'+str(file_name)+'_person_kp.pt'
        output_person_adj = data_folder_dst+'/'+str(file_name)+'_person_adj.pt'

        torch.save(torch.stack(sekeleton_person_gt), output_person_kp)
        torch.save(torch.stack(adjacency_person_gt), output_person_adj)
        
        print("saved:", output_person_kp)

            # data_info = {
            #     "file_name":file_name,
            #     "label_emotion": label_emotion,
            #     "persons": persons,
            #     "faces":faces,
            #     }
            # keep_files_names.append(data_info)
        #break

    with open(csv_file_dst, 'w') as file:
        json.dump(keep_files_names, file, indent=4)




In [ ]:
# csv_file_src =  "...../GAF_3.0/train_annotation_all_vitpose_kp.json"
# data_folder_src= '......./GAF_3.0/Train'

# csv_file_dst =  "......./GAF_3.0/train_annotation_all_vitpose_skeleton.json"
# data_folder_dst= '......./GAF_3.0/Train_skeleton_tensor'

# SaveSkeletons(data_folder_src,csv_file_src, data_folder_dst, csv_file_dst, score=0.1)